In [1]:
import lightgbm
import pandas as pd
import numpy as np
import os
import sys
from dotenv import load_dotenv
import json
import mlflow
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, TimeSeriesSplit
import time

In [2]:
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
from about_data.data_load import load_df

In [3]:
pd.set_option("display.max_columns", 100)
full_df = load_df(data_path)

In [4]:
sys.path.append(src_path)
from features.engineering import create_d_features
from about_data.split import temporal_split

json_path = os.path.join(data_path, r"processed/split_info.json")
with open(json_path, "r") as f:
    split_info = json.load(f)
    
train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

train, val, test = temporal_split(full_df, train_end, val_end)

In [5]:
len(create_d_features(train).columns)

39

In [6]:
map_dfs = {"train": train, "val": val, "test": test}
d_features_dfs = {}
for name, sample_df in map_dfs.items():
    d_features_dfs[name] = create_d_features(sample_df)

In [7]:
y_datasets = {}

for name, sample_df in map_dfs.items():
    y_datasets[name] = sample_df['isFraud']

In [8]:
X_train = d_features_dfs["train"].copy()
X_val = d_features_dfs["val"].copy()

y_train = y_datasets["train"]
y_val = y_datasets["val"]

In [9]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', is_unbalance=False, random_state=42, n_jobs=-1)

In [10]:
from model.preprocessor_pipe_evalueate import get_preprocessor, evaluate_model, create_pipeline

pipe = create_pipeline(model, get_preprocessor(X_train))

In [11]:
param_grid = param_distributions = {
    "model__n_estimators": [300, 500, 700, 1000],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "model__num_leaves": [15, 31, 63, 127],
    "model__max_depth": [-1, 5, 7, 10, 15],
    "model__min_child_samples": [20, 50, 100, 200],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.1, 0.5, 1.0],
    "model__reg_lambda": [0, 0.1, 0.5, 1.0, 5.0]
}

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tscv = TimeSeriesSplit(n_splits=5)
grid = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    scoring="average_precision",
    cv = tscv,
    n_iter=40,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True)

In [13]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-tuning")

with mlflow.start_run(run_name="LightGBM_RandomizedSearch"):
    start = time.time()

    grid.fit(X_train, y_train)

    training_time = time.time() - start

    best_model = grid.best_estimator_

    metrics = evaluate_model(grid, X_val, y_val)

    mlflow.log_params({
        "model": "lightgbm",
        "search_method": "RandomizedSearchCV",
        "n_iter": 40,
        "cv": "TimeSeriesSplit",
        "n_splits": 5,
        "scoring": "average_precision",
        "feature_count": X_train.shape[1],
        "n_train": len(X_train),
        "n_validation": len(X_val)
    })

    mlflow.log_metric("best_cv_pr_auc", grid.best_score_)

    mlflow.log_metrics(metrics)

    mlflow.log_metric("training_time_seconds", training_time
    )

    mlflow.log_params({f"best_{key}": value for key, value in grid.best_params_.items()
    })


Fitting 5 folds for each of 40 candidates, totalling 200 fits
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.084641 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11149
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4096
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


ValueError: Found input variables with inconsistent numbers of samples: [413378, 88581]